##### Import the required modules and configure the system path to locate them

In [1]:
import sys
sys.path.append("../../utils")
import networkx
import astra_sim_sdk.astra_sim_sdk as astra_sim_kit
from astra_sim import AstraSim, Collective, NetworkBackend
from infragraph.infragraph_service import InfraGraphService
from infragraph.blueprints.devices.nvidia.dgx import NvidiaDGX, DgxProfile
from infragraph import Infrastructure



##### Call the AstraSim client helper with the server endpoint and tag to connect to the ASTRA-sim gRPC server, initialize the SDK, and create a tagged folder for configs, results, and logs

In [2]:
astra = AstraSim(server_endpoint = "172.17.0.2:8989", tag = "analytical_dgx_device")

Resetting test directory
All contents of the folder /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_dgx_device have been deleted.
Successfully connected to gRPC server at 172.17.0.2:8989


##### Get all available DGX variants

In [3]:
from typing import get_args
print(get_args(DgxProfile))

('dgx1', 'dgx2', 'dgx_a100', 'dgx_h100', 'dgx_gb200')


##### Create a Nvidia DGX device fabric using infragraph device blueprint

In [4]:
server = NvidiaDGX()
infrastructure = Infrastructure()
infrastructure.devices.append(server)
infrastructure.instances.add(name=server.name, device=server.name, count=1)
astra.configuration.infragraph.infrastructure.deserialize(infrastructure.serialize())
print(astra.configuration.infragraph.infrastructure)

devices:
- components:
  - choice: cpu
    count: 2
    description: AMD EPYC 9654 (Genoa)
    name: cpu
  - choice: xpu
    count: 8
    description: NVIDIA H100 / H200 SXM5
    name: xpu
  - choice: switch
    count: 4
    description: NVIDIA NVSwitch
    name: nvlsw
  - choice: switch
    count: 3
    description: Broadcom PCIe Gen5 Switch
    name: pciesw
  - choice: custom
    count: 8
    custom:
      type: pcie_slot
    description: PCIe Gen5 x16 slots (ConnectX / BlueField)
    name: pciesl
  - choice: nic
    count: 8
    description: NVIDIA ConnectX / BlueField
    name: cx7
  description: NVIDIA DGX System
  edges:
  - ep1:
      component: cpu
    ep2:
      component: cpu
    link: cpu_fabric
    scheme: many2many
  - ep1:
      component: cpu[0]
    ep2:
      component: pciesl[0:3]
    link: pcie
    scheme: many2many
  - ep1:
      component: cpu[1]
    ep2:
      component: pciesl[4:7]
    link: pcie
    scheme: many2many
  - ep1:
      component: cpu[0]
    ep2:
    

##### Initialize the Infragraph service, display the fabric topology, and retrieve/set the total number of NPUs to generate the collective

In [5]:
service = InfraGraphService()
service.set_graph(infrastructure)
total_npus = service.get_component(device=server, type="xpu").count
g = service.get_networkx_graph()
print(networkx.write_network_text(g, vertical_chains=True))

╙── dgx_h100.0.cx7.1
    │
    dgx_h100.0.pciesl.1
    ├── dgx_h100.0.cpu.0
    │   ├── dgx_h100.0.cpu.1
    │   │   ├── dgx_h100.0.pciesl.4
    │   │   │   ├── dgx_h100.0.xpu.4
    │   │   │   │   ├── dgx_h100.0.nvlsw.0
    │   │   │   │   │   ├── dgx_h100.0.pciesw.2 ─ dgx_h100.0.cpu.0
    │   │   │   │   │   │   ├── dgx_h100.0.nvlsw.1 ─ dgx_h100.0.xpu.4
    │   │   │   │   │   │   │   ├── dgx_h100.0.xpu.0 ─ dgx_h100.0.nvlsw.0
    │   │   │   │   │   │   │   │   ├── dgx_h100.0.pciesl.0 ─ dgx_h100.0.cpu.0
    │   │   │   │   │   │   │   │   │   │
    │   │   │   │   │   │   │   │   │   dgx_h100.0.cx7.0
    │   │   │   │   │   │   │   │   ├── dgx_h100.0.nvlsw.2 ─ dgx_h100.0.pciesw.2, dgx_h100.0.xpu.4
    │   │   │   │   │   │   │   │   │   ├── dgx_h100.0.xpu.1 ─ dgx_h100.0.pciesl.1, dgx_h100.0.nvlsw.0, dgx_h100.0.nvlsw.1
    │   │   │   │   │   │   │   │   │   │   │
    │   │   │   │   │   │   │   │   │   │   dgx_h100.0.nvlsw.3 ─ dgx_h100.0.pciesw.2, dgx_h100.0.xpu.0, dgx_h100.0.xpu.4
 

##### Generate workload execution traces for each rank and set the required data size for AstraSim configuration

In [6]:
astra.configuration.common_config.workload = astra.generate_collective(collective=Collective.ALLREDUCE, coll_size= 1 *1024*1024, npu_range=[0, total_npus])

Generated 8 et in /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_dgx_device/configuration/workload


##### Configure ASTRA-sim system config

In [7]:
astra.configuration.common_config.system.scheduling_policy = astra.configuration.common_config.system.LIFO
astra.configuration.common_config.system.endpoint_delay = 10
astra.configuration.common_config.system.active_chunks_per_dimension = 1
astra.configuration.common_config.system.all_gather_implementation = [astra.configuration.common_config.system.RING]
astra.configuration.common_config.system.all_to_all_implementation = [astra.configuration.common_config.system.DIRECT]
astra.configuration.common_config.system.all_reduce_implementation = [astra.configuration.common_config.system.ONERING]
astra.configuration.common_config.system.collective_optimization = astra.configuration.common_config.system.LOCALBWAWARE
astra.configuration.common_config.system.local_mem_bw = 1600

##### Configure ASTRA-sim remote memory configuration

In [8]:
astra.configuration.common_config.remote_memory.memory_type = astra.configuration.common_config.remote_memory.NO_MEMORY_EXPANSION
print(astra.configuration.common_config.remote_memory)

memory_type: NO_MEMORY_EXPANSION
remote_mem_bw: 0
remote_mem_latency: 0



##### Set ASTRA-sim network backend to ANALYTICAL_CONGESTION_AWARE

In [9]:
astra.configuration.network_backend.analytical_congestion_aware.topology.choice = astra.configuration.network_backend.analytical_congestion_aware.topology.INFRAGRAPH

##### Adding ASTRA-sim - Infragraph specific annotation

In [10]:
host_device_spec = astra_sim_kit.AnnotationDeviceSpecifications()
host_device_spec.device_bandwidth_gbps = 100
host_device_spec.device_latency_ms = 0.05
host_device_spec.device_name = server.name
host_device_spec.device_type = "host"
astra.configuration.infragraph.annotations.device_specifications.append(host_device_spec)

##### Configure ASTRA-sim cmd parameters

In [11]:
astra.configuration.common_config.cmd_parameters.comm_scale = 1
astra.configuration.common_config.cmd_parameters.injection_scale = 1
astra.configuration.common_config.cmd_parameters.rendezvous_protocol = False

#### Start the simulation by specifying the network backend

In [12]:
astra.run_simulation(NetworkBackend.ANALYTICAL_CONGESTION_AWARE)

Generating Configuration ZIP now
output_path: /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_dgx_device/config.zip
folder_path: /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_dgx_device/configuration/workload/..
pack_zip complete


message: 'Configuration applied successfully. warnings: Unable to generate communicator
  group message from schema - communicator group configuration empty'

message: Simulation started successfully

astra-sim server Status: running
Transferring Files from ASTRA-sim server
All files downloaded Successfully
Simulation completed


##### Save infragraph as a yaml

In [ ]:
import yaml
import os
from common import FileFolderUtils
with open(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR,"../infrastructure","analytical_dgx_device.yaml"),"w") as f:
    data = infrastructure.serialize("dict")
    yaml.dump(data, f, default_flow_style=False, indent=4)

print("saved yaml to:", os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR,"..","analytical_dgx_device.yaml"))

saved yaml to: /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_dgx_device/output/../analytical_dgx_device.yaml


: 